# ARC-AGI-2 / ARC-AGI-3 Solver
## LLM-Powered Ensemble with Chain-of-Thought + Verification

**Approach:**
1. Rich task analysis (colors, objects, symmetry, transformations)
2. 5-prompt ensemble with diverse reasoning strategies
3. Majority voting
4. Self-verification pass

**Model:** Claude Opus (Anthropic) — best reasoning on ARC-style spatial tasks

In [ ]:
# Install dependencies
!pip install anthropic -q
!pip install numpy -q

In [ ]:
import os
# Set your API key (use Kaggle Secrets for the competition)
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['ANTHROPIC_API_KEY'] = secrets.get_secret('ANTHROPIC_API_KEY')

In [ ]:
# Copy source files (upload arc_solver/src/ to your Kaggle dataset)
import sys
sys.path.insert(0, '/kaggle/input/arc-solver-src/src')

In [ ]:
from arc_types import ARCTask
from solver import ARCSolver, generate_kaggle_submission
from evaluator import load_tasks_from_dir, evaluate_solver_results, print_evaluation_report

# Paths
DATA_DIR = '/kaggle/input/arc-prize-2026/arc-agi_evaluation_challenges'
OUTPUT_CSV = '/kaggle/working/submission.csv'

In [ ]:
# Initialize solver
solver = ARCSolver(
    provider='anthropic',
    model='claude-opus-4-6',
    n_ensemble=5,           # 5 diverse prompts per test
    do_verification=True,   # Self-correction pass
    verify_rounds=1,
    verbose=True,
)

print('Solver ready!')

In [ ]:
# Run on full evaluation set
results = solver.solve_dataset(
    data_dir=DATA_DIR,
    output_path='/kaggle/working/results.json'
)

In [ ]:
# Generate Kaggle submission
generate_kaggle_submission(solver, DATA_DIR, OUTPUT_CSV)
print(f'Submission ready: {OUTPUT_CSV}')

In [ ]:
# Show sample predictions
import json
with open('/kaggle/working/results.json') as f:
    res = json.load(f)

tasks_shown = 0
for task_id, task_result in res.get('tasks', {}).items():
    for test_idx, tres in task_result.items():
        if tres.get('prediction'):
            print(f'Task: {task_id} | Confidence: {tres["confidence"]:.0%}')
            print('Prediction:')
            for row in tres['prediction']:
                print(' ', ' '.join(str(c) for c in row))
            print()
            tasks_shown += 1
            if tasks_shown >= 3:
                break
    if tasks_shown >= 3:
        break